In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
import sys
import os
from datetime import datetime
from pyspark.sql.functions import col, date_format

# --------------------------------------------------
# 1. Widgets & Setup
# --------------------------------------------------
dbutils.widgets.text("customer_id", "")
dbutils.widgets.text("object_name", "")
dbutils.widgets.text("source_system", "salesforce")
dbutils.widgets.text("sf_catalog_table", "")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
sf_catalog_table = dbutils.widgets.get("sf_catalog_table")
load_type = "incremental"

# --------------------------------------------------
# 2. Import Configuration dynamically
# --------------------------------------------------
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
abs_project_root = os.path.abspath(project_root)

if abs_project_root not in sys.path:
    sys.path.append(abs_project_root)

from utils.config import DataLakeConfig
from utils.pipeline_metrics import PipelineMetrics

# Instantiate the config
config = DataLakeConfig(source_system, customer_id, object_name, load_type)

# ── Import and initialize metrics ──
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "SF_Incremental_Load",   # change per notebook
    task_key       = "run_job_B_sf_ingestion", # match the YAML task_key
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()   # ← writes RUNNING row immediately (<100ms)

# Optimize shuffle for small clusters
spark.conf.set("spark.sql.shuffle.partitions", 4)

# Generate Timestamped path via Config Helper
now = datetime.now()
incremental_path = config.get_landing_zone_timestamped_path(source_system, customer_id, object_name, load_type, now)

print(f"🚀 Starting Incremental Load to: {incremental_path}")

# --------------------------------------------------
# 3. Get the last watermark
# --------------------------------------------------
try:
    watermark_df = spark.sql(f"""
    SELECT last_processed_timestamp
    FROM ingestion_metadata.watermarks
    WHERE source_system='{source_system}'
    AND object_name='{object_name}'
    """)
    rows = watermark_df.collect()
    if len(rows) == 0:
        raise Exception("Watermark missing.")
    watermark = rows[0][0]
except Exception as e:
    raise Exception(f"❌ Watermark not initialized. Please run historical load first. Error: {str(e)}")

print(f"📍 Last Watermark: {watermark}")

# 4. Pre-Fetch: Query Salesforce FIRST to get the NEW max timestamp
new_ts_df = spark.sql(f"""
SELECT MAX(SystemModstamp) as max_ts
FROM {sf_catalog_table}
WHERE SystemModstamp > TIMESTAMP('{watermark}')
""")

new_ts = new_ts_df.first()[0]

if new_ts is None:
    print("✅ No new records found in Salesforce. Exiting gracefully.")
    dbutils.notebook.exit("0")

print(f"🎯 New records found up to: {new_ts}")

# 5. Pull the data using BOTH bounds
df_incremental = spark.sql(f"""
SELECT *
FROM {sf_catalog_table}
WHERE SystemModstamp > TIMESTAMP('{watermark}')
  AND SystemModstamp <= TIMESTAMP('{new_ts}')
""")

# Format the Timestamp to exact String format 
df_formatted = df_incremental.withColumn("SystemModstamp", date_format(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))

try:
    # 6. Write to S3 using config path
    (
        df_formatted
        .repartition(4)
        .write
        .mode("append")
        .format("parquet")
        .option("compression", "snappy")
        .save(incremental_path)
    )
    print("💾 Write to S3 completed successfully.")
    
except Exception as e:
    print(f"❌ Failed during write operation: {str(e)}")
    raise e

# 7. Update Watermark with 1-Minute Lookback
print("Updating watermark table safely using MERGE...")

spark.sql(f"""
    MERGE INTO ingestion_metadata.watermarks AS target
    USING (
        SELECT '{source_system}' AS source_system, 
               '{object_name}' AS object_name, 
               TIMESTAMP('{new_ts}') - INTERVAL 1 MINUTE AS last_processed_timestamp
    ) AS source
    ON target.source_system = source.source_system AND target.object_name = source.object_name
    WHEN MATCHED THEN
        UPDATE SET target.last_processed_timestamp = source.last_processed_timestamp
    WHEN NOT MATCHED THEN
        INSERT (source_system, object_name, last_processed_timestamp)
        VALUES (source.source_system, source.object_name, source.last_processed_timestamp)
""")

print(f"✅ Watermark updated safely (minus 1 minute).")


In [0]:
# ── LINE 3 & 4: At the end, wrap work in try/except ──
try:
    # ... your existing notebook code here (unchanged) ...
    total_rows = df_formatted.count()  # however you count rows
    
    pm.save(status="SUCCESS", rows_processed=total_rows)  # LINE 3
    
except Exception as e:
    pm.save(status="FAILED", error_reason=str(e))          # LINE 4
    raise